### Imports

In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import os
os.chdir('C:\Kaggle_Competition\Playground\S6E4-Irrigation-Need')
import json
import warnings
import gc
import numpy as np
import skops.io as sio
import pandas as pd
import matplotlib.pyplot as plt
import config
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.inspection import permutation_importance
from src.utils import (
    read_csv_file, 
    save_csv_file, 
    agnostic_bacc_scorer,
    compute_bacc,
    compute_logloss,
    compute_recall_per_class,
    reduce_mem_usage
)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from category_encoders import CountEncoder, SumEncoder, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import mlflow
from tqdm.notebook import tqdm
from pytabkit import (
	RealMLP_TD_Classifier,
	TabM_D_Classifier,
	Resnet_RTDL_D_Classifier,
	CatBoost_TD_Classifier,
	LGBM_TD_Classifier,
	XGB_TD_Classifier
)
from src.fe import *
pd.set_option('display.max_columns', None)

import logging
logging.getLogger("mlflow.utils.environment").setLevel(logging.ERROR)

### Prepare data

In [26]:
train_raw = read_csv_file(r'data\raw\train.csv')
test_raw = read_csv_file(r'data\raw\test.csv')

# Train and test ids
train_ids = train_raw['id']
test_ids = test_raw['id']

n_classes = 3

Reading file from: data\raw\train.csv
Reading file from: data\raw\test.csv


In [27]:
# X and y split
X = train_raw.drop(['id', 'irrigation_need'], axis=1)
y = train_raw['irrigation_need'].map(config.TARGET_MAP)
classes = np.unique(y)

# Test data
test_data = test_raw.drop('id', axis=1)

In [28]:
# Reduce memory size of data
X = reduce_mem_usage(X)
test_data = reduce_mem_usage(test_data)

Memory before: 354.06 MB
Memory after: 327.63 MB
Reduced by: 26.44 MB (7.5%)
Memory before: 151.74 MB
Memory after: 140.41 MB
Reduced by: 11.33 MB (7.5%)


In [29]:
RUN = 'v1'

### Logistic Regression

In [30]:
# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [31]:
EXP_NAME = "LOGISTIC"
RUN_NAME = f"Logistic_seed{config.SEED}_{RUN}"

skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

final_oof_preds = np.zeros((len(X), n_classes))
final_test_preds = np.zeros((len(test_data), n_classes))
feature_importance = []
bacc_scores = []
ll_scores = []

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    # -------- STATIC LOGS --------
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")
    mlflow.log_params({
		"seed": config.SEED,
		"n_folds": config.N_FOLDS,
		"n_cols": X.shape[1],
		**config.LOGISTIC_PARAMS
	})

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]

        # -------- PIPELINE --------
        preprocess = ColumnTransformer(
            transformers=[
                ('ohe', OneHotEncoder(drop='first', sparse_output=False, dtype=int), cat_cols),
                ('scaler', StandardScaler(), num_cols)
			],
		remainder='passthrough').set_output(transform='pandas')

        model = Pipeline(
            steps=[
                ('prep', preprocess),
                ('model', LogisticRegression(**config.LOGISTIC_PARAMS))
			]
		)

        model.fit(X_train, y_train)

        # -------- FOLD RUN --------
        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):
            mlflow.sklearn.log_model(model, name="model", serialization_format="skops")

            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict_proba(test_data)
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring='balanced_accuracy',
                n_repeats=1,
                n_jobs=-1,
                random_state=config.SEED
            )
            feature_importance.append(fi.importances_mean)

    # -------- FINAL METRICS --------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    metrics = {
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": np.std(bacc_scores),
        "std_logloss": np.std(ll_scores),
        "recall_0": recall_per_class[0],
        "recall_1": recall_per_class[1],
        "recall_2": recall_per_class[2]
    }

    mlflow.log_metrics(metrics)
    feature_names = model.named_steps['prep'].get_feature_names_out()
    mlflow.log_text("\n".join(feature_names), "artifacts/preprocessed_cols.txt")

    # -------- FEATURE IMPORTANCE --------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)

    fi_df = pd.DataFrame({
        "feature": X_valid.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")
    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)

# -------- SAVE FILES --------
oof_df = pd.DataFrame(final_oof_preds, columns=[str(c) for c in classes])
oof_df.insert(0, "id", train_ids)
save_csv_file(oof_df, os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv"))

test_df = pd.DataFrame(final_test_preds, columns=[str(c) for c in classes])
test_df.insert(0, "id", test_ids)
save_csv_file(test_df, os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv"))

Fold 1 | Balanced Acc: 0.85016 | LogLoss: 0.35586
Fold 2 | Balanced Acc: 0.84849 | LogLoss: 0.36034
Fold 3 | Balanced Acc: 0.85001 | LogLoss: 0.35803
Fold 4 | Balanced Acc: 0.85081 | LogLoss: 0.36093
Fold 5 | Balanced Acc: 0.84818 | LogLoss: 0.35792
Saving file to: artifacts\oof\Logistic_seed42_v1_oof_proba.csv
Saving file to: artifacts\test_proba\Logistic_seed42_v1_test_proba.csv


In [39]:
from shap import LinearExplainer
import shap

In [36]:
prep = model.named_steps['prep']
my_model = model.named_steps['model']

In [38]:
X_train_tr = prep.transform(X_train)

In [41]:
expaliner = LinearExplainer(
    my_model, X_train_tr
)

In [44]:
expaliner.expected_value

array([ 1.76799725,  1.93385239, -3.70184964])

### Catboost Core

In [168]:
# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [ ]:
EXP_NAME = "CATBOOST"
RUN_NAME = f"Catgbm_seed{config.SEED}_{RUN}"

skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

final_oof_preds = np.zeros((len(X), n_classes))
final_test_preds = np.zeros((len(test_data), n_classes))
feature_importance = []
bacc_scores = []
ll_scores = []

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    # -------- STATIC LOGS --------
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")
    mlflow.log_params({
		"seed": config.SEED,
		"n_folds": config.N_FOLDS,
		"n_cols": X.shape[1],
		**config.CATBOOST_PARAMS
	})

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx]
        X_valid, y_valid = X.iloc[valid_idx].copy(), y.iloc[valid_idx]

		# ------ Pipeline -----
        train_pool = cb.Pool(data=X_train, label=y_train, cat_features=cat_cols, thread_count=-1)
        valid_pool = cb.Pool(data=X_valid, label=y_valid, cat_features=cat_cols, thread_count=-1)
        test_pool = cb.Pool(data=test_data, cat_features=cat_cols, thread_count=-1)

        model = cb.train(
            train_pool,
            config.CATBOOST_PARAMS,
            eval_set=valid_pool,
            plot=True
		)

        # -------- FOLD RUN --------
        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):
            mlflow.catboost.log_model(model, name="model")

            oof_preds = model.predict(valid_pool, prediction_type='Probability')
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict(test_pool, prediction_type='Probability')
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=1,
                n_jobs=-1,
                random_state=config.SEED
            )
            feature_importance.append(fi.importances_mean)

    # -------- FINAL METRICS --------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    metrics = {
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": np.std(bacc_scores),
        "std_logloss": np.std(ll_scores),
        "recall_0": recall_per_class[0],
        "recall_1": recall_per_class[1],
        "recall_2": recall_per_class[2]
    }

    mlflow.log_metrics(metrics)

    # -------- FEATURE IMPORTANCE --------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)

    fi_df = pd.DataFrame({
        "feature": X_valid.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")
    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)

# -------- SAVE FILES --------
oof_df = pd.DataFrame(final_oof_preds, columns=[str(c) for c in classes])
oof_df.insert(0, "id", train_ids)
save_csv_file(oof_df, os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv"))

test_df = pd.DataFrame(final_test_preds, columns=[str(c) for c in classes])
test_df.insert(0, "id", test_ids)
save_csv_file(test_df, os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv"))

###  XGBoost TD

In [213]:
for df in [X, test_data]:
	df["soil_lt_25"] = (df["soil_moisture"] < 25).astype(int)
	df["temp_gt_30"] = (df["temperature_c"] > 30).astype(int)
	df["rain_lt_300"] = (df["rainfall_mm"] < 300).astype(int)
	df["wind_gt_10"] = (df["wind_speed_kmh"] > 10).astype(int)

In [214]:
keep = [
    "crop_growth_stage",
    "soil_lt_25",
    "mulching_used",
    "temperature_c",
    "wind_gt_10",
    "rainfall_mm",
    "wind_speed_kmh",
    "soil_moisture",
    "rain_lt_300",
    "temp_gt_30",
    "humidity",
    "previous_irrigation_mm",
]

In [215]:
X = X[keep]
test_data = test_data[keep]

In [216]:
# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [217]:
EXP_NAME = "XGBOOST_TD"
RUN_NAME = f"xgbm_TD_seed{config.SEED}_{RUN}"

skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

final_oof_preds = np.zeros((len(X), n_classes))
final_test_preds = np.zeros((len(test_data), n_classes))
feature_importance = []
bacc_scores = []
ll_scores = []

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    # -------- STATIC LOGS --------
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")
    mlflow.log_params({
		"seed": config.SEED,
		"n_folds": config.N_FOLDS,
		"n_cols": X.shape[1],
		**config.XGB_TD_PARAMS
	})

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx]
        X_valid, y_valid = X.iloc[valid_idx].copy(), y.iloc[valid_idx]

		# ------ Pipeline -----

        model = XGB_TD_Classifier(**config.XGB_TD_PARAMS)
        model.fit(X_train, y_train, X_valid, y_valid, cat_col_names=cat_cols)

        # -------- FOLD RUN --------
        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):
            mlflow.sklearn.log_model(model, name="model")

            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict_proba(test_data)
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring='balanced_accuracy',
                n_repeats=1,
                n_jobs=-1,
                random_state=config.SEED
            )
            feature_importance.append(fi.importances_mean)

    # -------- FINAL METRICS --------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    metrics = {
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": np.std(bacc_scores),
        "std_logloss": np.std(ll_scores),
        "recall_0": recall_per_class[0],
        "recall_1": recall_per_class[1],
        "recall_2": recall_per_class[2]
    }

    mlflow.log_metrics(metrics)

    # -------- FEATURE IMPORTANCE --------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)

    fi_df = pd.DataFrame({
        "feature": X_valid.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")
    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)

# -------- SAVE FILES --------
oof_df = pd.DataFrame(final_oof_preds, columns=[str(c) for c in classes])
oof_df.insert(0, "id", train_ids)
save_csv_file(oof_df, os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv"))

test_df = pd.DataFrame(final_test_preds, columns=[str(c) for c in classes])
test_df.insert(0, "id", test_ids)
save_csv_file(test_df, os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv"))

2026/04/18 01:14:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 01:14:13 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Fold 1 | Balanced Acc: 0.96252 | LogLoss: 0.06047


2026/04/18 01:14:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 01:14:42 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Fold 2 | Balanced Acc: 0.96417 | LogLoss: 0.05841


2026/04/18 01:15:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 01:15:30 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


...

Fold 4 | Balanced Acc: 0.96333 | LogLoss: 0.05688


2026/04/18 01:16:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 01:16:45 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Fold 5 | Balanced Acc: 0.96309 | LogLoss: 0.05687


c:\Users\aryan\miniconda3\envs\kaggle\Lib\site-packages\sklearn\metrics\_classification.py:310: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Saving file to: artifacts\oof\xgbm_TD_seed42_v4_oof_proba.csv
Saving file to: artifacts\test_proba\xgbm_TD_seed42_v4_test_proba.csv
